# DWBI Data Transformation (ETL Phase)
This notebook handles the cleaning, data type conversion, and feature engineering for the A/L dataset.

In [ ]:
import pandas as pd
import numpy as np
import io
from google.colab import files

print("Please upload your CSV file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Data loaded successfully!")

In [ ]:
print("Starting data transformations...")

# 1. Handle Missing & Invalid Values
df.replace(['-', 'Absent', 'Major error'], np.nan, inplace=True)

# 2. Data Type Conversions
df['Zscore'] = pd.to_numeric(df['Zscore'], errors='coerce')
df['district_rank_num'] = df['district_rank'].astype(str).str.extract(r'(\d+)').astype(float)
df['island_rank_num'] = df['island_rank'].astype(str).str.extract(r'(\d+)').astype(float)

# 3. Derived Columns (Feature Engineering)
month_map = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6, 
             'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}

df['birth_month_num'] = df['birth_month'].map(month_map)
df['birth_date'] = pd.to_datetime({
    'year': df['birth_year'],
    'month': df['birth_month_num'],
    'day': df['birth_day']
}, errors='coerce')

passing_grades = ['A', 'B', 'C', 'S']
df['passed_sub1'] = df['sub1_r'].isin(passing_grades)
df['passed_sub2'] = df['sub2_r'].isin(passing_grades)
df['passed_sub3'] = df['sub3_r'].isin(passing_grades)
df['is_fully_qualified'] = df['passed_sub1'] & df['passed_sub2'] & df['passed_sub3']

conditions = [
    (df['Zscore'] >= 1.5),
    (df['Zscore'] >= 0.0) & (df['Zscore'] < 1.5),
    (df['Zscore'] < 0.0)
]
choices = ['High Performer', 'Average Performer', 'Below Average']
df['performance_band'] = np.select(conditions, choices, default='Unknown')

# 4. Standardizing Text
df['stream'] = df['stream'].astype(str).str.title()
df['gender'] = df['gender'].astype(str).str.capitalize()
df.drop(columns=['birth_month_num'], inplace=True)

print("Transformations complete! Here is a preview:")
df[['stream', 'gender', 'birth_date', 'Zscore', 'performance_band', 'is_fully_qualified']].head(10)

In [ ]:
# Download the cleaned data
df.to_csv('final_transformed_data.csv', index=False)
files.download('final_transformed_data.csv')